# system_tai Phase 1.5C — Kaggle compatibility calibration

This notebook discovers the private Dataset_AIC2026 attachment dynamically, including nested `/kaggle/input/datasets/<owner-or-runtime-id>/<dataset-root>` layouts. It never copies source videos, keyframes, NPY files, or dataset directories. Local unit tests are synthetic mechanics evidence only; outputs produced here become real BTC evidence only when these cells are executed against the attached private dataset. CLIP compatibility remains UNVERIFIED unless the multi-video identification gate succeeds.

## 1. Environment and attached-input inspection

In [ ]:
import csv
import json
import platform
import subprocess
import sys
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")
REPO_ROOT = Path("/kaggle/working/AI_Challenge_HCM")
SYSTEM_ROOT = REPO_ROOT / "systems/system_tai"
OUTPUT_ROOT = Path("/kaggle/working/system_tai_outputs/calibration")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
attached_children = sorted(str(path) for path in INPUT_ROOT.iterdir() if path.is_dir())
environment = {
    "python": sys.version,
    "platform": platform.platform(),
    "input_children": attached_children,
    "repository_exists": REPO_ROOT.is_dir(),
}
print(json.dumps(environment, indent=2))

## 2. Repository/package setup instructions

Clone the repository to `/kaggle/working/AI_Challenge_HCM` before running this notebook. No credentials or tokens belong in the notebook. The editable install affects code only; it does not copy the attached dataset.

In [ ]:
if not SYSTEM_ROOT.is_dir():
    raise FileNotFoundError(f"Repository package not found: {SYSTEM_ROOT}")
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(SYSTEM_ROOT)], check=True)
yaml = __import__("yaml")

CONFIG_PATH = SYSTEM_ROOT / "configs/phase_1_5_kaggle.example.yaml"
config = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
assert Path(config["input_root"]) == INPUT_ROOT
assert Path(config["report_output_directory"]) == OUTPUT_ROOT

## 3. Dynamic nested dataset-root discovery

Discovery starts at `/kaggle/input` and does not assume a direct child or fixed dataset slug.

In [ ]:
def run_json_command(arguments, *, allow_nonzero=False):
    completed = subprocess.run(arguments, text=True, capture_output=True)
    if completed.stderr.strip():
        print(completed.stderr)
    if completed.returncode and not allow_nonzero:
        raise RuntimeError(
            f"Command failed ({completed.returncode}): {arguments}\n{completed.stdout}"
        )
    return completed.returncode, json.loads(completed.stdout)


discovery_script = SYSTEM_ROOT / "scripts/discover_kaggle_inputs.py"
manifests = {}
for video_id in config["requested_video_ids"]:
    output_path = OUTPUT_ROOT / f"discovery_{video_id}.json"
    command = [
        sys.executable,
        str(discovery_script),
        "--input-root",
        str(INPUT_ROOT),
        "--video-id",
        video_id,
        "--output",
        str(output_path),
    ]
    for hint in config["discovery_hints"]:
        command.extend(["--hint", hint])
    _, manifests[video_id] = run_json_command(command)
print(json.dumps({key: value["dataset_root"] for key, value in manifests.items()}, indent=2))

## 4. Discovery of three calibration videos

In [ ]:
required_video_ids = {"L21_V001", "L21_V002", "L22_V001"}
missing_video_ids = sorted(required_video_ids - manifests.keys())
if missing_video_ids:
    raise RuntimeError(f"Missing required calibration videos: {missing_video_ids}")
discovery_summary = {
    video_id: {
        "status": manifest["status"],
        "dataset_root": manifest["dataset_root"],
        "resolved_artifact_count": sum(
            value is not None for value in manifest["artifacts"].values()
        ),
    }
    for video_id, manifest in manifests.items()
}
print(json.dumps(discovery_summary, indent=2))

## 5. Real-input and mapping-rounding audits

The temporary catalog below is a decoded-metadata calibration input, not an authoritative team catalog. It records the current zero-based working interpretation and is written only to the permitted report directory. The mapping audit compares Decimal floor, binary-float truncation, and Decimal nearest as diagnostic numeric models without changing `frame_idx` or gating mapping validity.

In [ ]:
import cv2

audit_script = SYSTEM_ROOT / "scripts/audit_kis_inputs.py"
audit_reports = {}
rounding_reports = {}
for video_id, manifest in manifests.items():
    artifacts = manifest["artifacts"]
    capture = cv2.VideoCapture(artifacts["original_video"])
    if not capture.isOpened():
        raise RuntimeError(f"Cannot decode {video_id}")
    fps = float(capture.get(cv2.CAP_PROP_FPS))
    total_frames = int(round(capture.get(cv2.CAP_PROP_FRAME_COUNT)))
    capture.release()
    if fps <= 0 or total_frames <= 0:
        raise RuntimeError(
            f"Invalid decoded metadata for {video_id}: fps={fps}, frames={total_frames}"
        )
    catalog_path = OUTPUT_ROOT / f"derived_catalog_{video_id}.csv"
    with catalog_path.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(
            stream,
            fieldnames=[
                "video_id",
                "video_path",
                "fps",
                "duration_seconds",
                "total_frames",
                "frame_index_base",
            ],
        )
        writer.writeheader()
        writer.writerow(
            {
                "video_id": video_id,
                "video_path": artifacts["original_video"],
                "fps": fps,
                "duration_seconds": total_frames / fps,
                "total_frames": total_frames,
                "frame_index_base": "zero_based",
            }
        )
    report_path = OUTPUT_ROOT / f"input_audit_{video_id}.json"
    command = [
        sys.executable,
        str(audit_script),
        "--video-catalog",
        str(catalog_path),
        "--video-id",
        video_id,
        "--mapping-csv",
        artifacts["mapping_csv"],
        "--clip-npy",
        artifacts["clip_npy"],
        "--strict-video-path-check",
        "--output",
        str(report_path),
    ]
    expected_dimension = config["clip_identification"]["expected_dimension"]
    if expected_dimension is not None:
        command.extend(["--expected-dimension", str(expected_dimension)])
    _, audit_reports[video_id] = run_json_command(command)
    rounding_path = OUTPUT_ROOT / f"mapping_rounding_{video_id}.json"
    _, rounding_reports[video_id] = run_json_command(
        [
            sys.executable,
            str(SYSTEM_ROOT / "scripts/audit_mapping_rounding.py"),
            "--mapping-csv",
            artifacts["mapping_csv"],
            "--video-id",
            video_id,
            "--output",
            str(rounding_path),
        ]
    )
print(
    json.dumps(
        {
            video_id: {
                "input_valid": audit_reports[video_id]["valid"],
                "rounding_rule": rounding_reports[video_id]["observed_rule_summary"],
                "decimal_floor_ratio": rounding_reports[video_id][
                    "frame_idx_equals_decimal_floor"
                ]["ratio"],
                "binary_float_truncation_ratio": rounding_reports[video_id][
                    "frame_idx_equals_binary_float_truncation"
                ]["ratio"],
                "nearest_offset_distribution": rounding_reports[video_id][
                    "decimal_nearest_minus_frame_idx_distribution"
                ],
            }
            for video_id in manifests
        },
        indent=2,
    )
)

## 6. Frame-index calibration

Mapping-coordinate validation preserves `frame_idx` exactly and checks raw-video bounds regardless of which diagnostic numeric-generation model matches. Timestamp diagnostics compare Decimal floor, binary-float truncation, and Decimal nearest. Visual comparison checks decoded `f-1`, `f`, and `f+1`; a `+1` visual winner is explained when it equals `decimal_round_half_up(pts_time * fps) - frame_idx`. No numeric or visual diagnostic changes the shared frame ID.

In [ ]:
calibration_cases = []
for video_id, manifest in manifests.items():
    artifacts = manifest["artifacts"]
    keyframe_source = artifacts["keyframe_directory"]
    if keyframe_source is None:
        examples = artifacts["keyframe_image_examples"]
        if len(examples) != 1:
            raise RuntimeError(
                f"{video_id} needs a keyframe directory for multi-sample calibration"
            )
        keyframe_source = examples[0]
    calibration_cases.append(
        {
            "video_id": video_id,
            "video_path": artifacts["original_video"],
            "mapping_csv": artifacts["mapping_csv"],
            "keyframes": keyframe_source,
            "sample_count": config["frame_calibration"]["sample_count"],
            "keyframe_orders": config["frame_calibration"]["explicit_keyframe_orders"],
            "offset_candidates": config["frame_calibration"]["frame_offset_candidates"],
            "superiority_margin": config["frame_calibration"]["superiority_margin"],
            "consistency_ratio": config["frame_calibration"]["consistency_ratio"],
            "decoder_agreement_tolerance": config["frame_calibration"][
                "decoder_agreement_tolerance"
            ],
        }
    )
batch_path = OUTPUT_ROOT / "frame_calibration_batch_manifest.json"
batch_path.write_text(json.dumps({"cases": calibration_cases}, indent=2) + "\n", encoding="utf-8")
calibration_output = OUTPUT_ROOT / "frame_mapping_calibration.json"
calibration_script = SYSTEM_ROOT / "scripts/calibrate_frame_mapping.py"
calibration_rc, calibration_report = run_json_command(
    [
        sys.executable,
        str(calibration_script),
        "--batch-manifest",
        str(batch_path),
        "--output",
        str(calibration_output),
    ],
    allow_nonzero=True,
)
print("return_code:", calibration_rc)
print(
    json.dumps(
        {
            "status": calibration_report["status"],
            "case_count": calibration_report.get("case_count"),
        },
        indent=2,
    )
)

## 7. Optional CLIP pipeline identification

Hugging Face is tested as an implementation interface for OpenAI CLIP, not as a separate model family. Generic SentenceTransformers are excluded. Missing packages, weights, or internet produce SKIPPED candidates and do not invalidate the frame/data audit.

In [ ]:
required_video_count = config["clip_identification"]["minimum_identification_videos"]
calibration_cases_by_video = {
    case["video_id"]: case for case in calibration_report.get("cases", [])
}
clip_gate_passed = (
    len(manifests) >= required_video_count
    and all(report.get("valid") for report in audit_reports.values())
    and all(
        calibration_cases_by_video.get(video_id, {})
        .get("mapping_coordinate_validation", {})
        .get("status")
        == "MAPPING_POLICY_PASSED"
        for video_id in manifests
    )
)
RUN_CLIP_IDENTIFICATION = False  # Set True only for a gated CLIP-only rerun.
clip_output = OUTPUT_ROOT / "clip_pipeline_identification.json"
if RUN_CLIP_IDENTIFICATION and not clip_gate_passed:
    clip_report = {
        "status": "GATE_BLOCKED",
        "reason": "All three videos must pass input, feature-row, and mapping gates",
        "text_query_encoder_implemented": False,
    }
    clip_output.write_text(
        json.dumps(clip_report, indent=2) + "\n", encoding="utf-8"
    )
elif RUN_CLIP_IDENTIFICATION:
    clip_cases = [
        {
            "video_id": case["video_id"],
            "mapping_csv": case["mapping_csv"],
            "clip_npy": manifests[case["video_id"]]["artifacts"]["clip_npy"],
            "keyframes": case["keyframes"],
            "sample_count": config["clip_identification"]["sample_count"],
            "keyframe_orders": config["clip_identification"]["explicit_keyframe_orders"],
            "expected_dimension": config["clip_identification"]["expected_dimension"],
        }
        for case in calibration_cases
    ]
    clip_batch = OUTPUT_ROOT / "clip_identification_batch_manifest.json"
    clip_batch.write_text(json.dumps({"cases": clip_cases}, indent=2) + "\n", encoding="utf-8")
    command = [
        sys.executable,
        str(SYSTEM_ROOT / "scripts/identify_btc_clip_pipeline.py"),
        "--batch-manifest",
        str(clip_batch),
        "--minimum-identification-videos",
        str(config["clip_identification"]["minimum_identification_videos"]),
        "--output",
        str(clip_output),
    ]
    for backend in config["clip_identification"]["enabled_candidate_backends"]:
        command.extend(["--backend", backend])
    if config["clip_identification"]["allow_model_download"]:
        command.append("--allow-model-download")
    _, clip_report = run_json_command(command, allow_nonzero=True)
else:
    clip_report = {
        "status": "NOT_RUN",
        "reason": "Phase 1.5C leaves compatibility UNVERIFIED until rerun",
        "text_query_encoder_implemented": False,
    }
    clip_output.write_text(json.dumps(clip_report, indent=2) + "\n", encoding="utf-8")
print(json.dumps(clip_report, indent=2))

## 8. Compact comparison tables

In [ ]:
import pandas as pd

frame_rows = []
for case in calibration_report.get("cases", []):
    video_id = case["video_id"]
    frame_rows.append(
        {
            "video_id": video_id,
            "input_valid": audit_reports[video_id]["valid"],
            "rounding_rule": rounding_reports[video_id]["observed_rule_summary"],
            "decimal_floor_ratio": rounding_reports[video_id][
                "frame_idx_equals_decimal_floor"
            ]["ratio"],
            "binary_float_truncation_ratio": rounding_reports[video_id][
                "frame_idx_equals_binary_float_truncation"
            ]["ratio"],
            "nearest_offset_distribution": json.dumps(
                rounding_reports[video_id][
                    "decimal_nearest_minus_frame_idx_distribution"
                ]
            ),
            "mapping_policy": case.get("mapping_coordinate_validation", {}).get("status"),
            "visual_status": case["status"],
            "sample_count": case.get("sample_count"),
            "decisive_sample_count": case.get("decisive_sample_count"),
            "ambiguous_sample_count": case.get("ambiguous_sample_count"),
            "sampled_visual_prediction_accuracy": case.get(
                "explained_decisive_ratio"
            ),
            "contradictory_decisive_sample_count": case.get(
                "contradictory_decisive_sample_count"
            ),
            "decoder_status": case.get("decoder_agreement", {}).get("status"),
        }
    )
frame_table = pd.DataFrame(frame_rows)
frame_table.to_csv(OUTPUT_ROOT / "frame_calibration_summary.csv", index=False)
display(frame_table)
clip_rows = [
    {"backend": backend, **result}
    for backend, result in clip_report.get("backend_summary", {}).items()
]
clip_table = pd.DataFrame(clip_rows)
clip_table.to_csv(OUTPUT_ROOT / "clip_pipeline_summary.csv", index=False)
display(clip_table)

## 9. Report-only output policy

Only JSON and CSV reports are saved under `/kaggle/working/system_tai_outputs/calibration/`. Dataset artifacts remain under `/kaggle/input` and are never copied.

In [ ]:
unexpected = [
    path
    for path in OUTPUT_ROOT.rglob("*")
    if path.is_file() and path.suffix.lower() not in {".json", ".csv"}
]
if unexpected:
    raise RuntimeError(f"Unexpected non-report outputs: {unexpected}")
print("Report files:")
for path in sorted(OUTPUT_ROOT.rglob("*")):
    if path.is_file():
        print(path)

## 10. Final status and next action summary

In [ ]:
final_status = {
    "code_and_tests": "VERIFY_SEPARATELY_IN_REPOSITORY",
    "real_btc_execution": "EXECUTED_IN_THIS_NOTEBOOK" if manifests else "NOT_RUN",
    "synthetic_unit_tests_are_btc_evidence": False,
    "mapping_rounding_audits": {
        video_id: report["observed_rule_summary"]
        for video_id, report in rounding_reports.items()
    },
    "frame_calibration": calibration_report.get("status", "NOT_RUN"),
    "clip_identification_gate_passed": clip_gate_passed,
    "clip_pipeline": clip_report.get("status", "NOT_RUN"),
    "text_query_encoder_compatibility": "UNVERIFIED",
    "next_action": (
        "Review all three mapping reports. Keep frame_id equal to frame_idx; "
        "do not implement semantic retrieval until CLIP compatibility is identified."
    ),
}
(OUTPUT_ROOT / "final_status.json").write_text(
    json.dumps(final_status, indent=2) + "\n", encoding="utf-8"
)
print(json.dumps(final_status, indent=2))